In [1]:
# 2.1 理论计算题

# 问题描述：
# 输入一张大小为 3 × 32 × 32（通道数 × 高 × 宽）的彩色图像。
# 通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 3 × 5 × 5。
# 设定填充（Padding）为 2，步幅（Stride）为 2。

def calculate_convolution_output(input_channels, input_height, input_width, 
                                num_kernels, kernel_channels, kernel_height, kernel_width,
                                padding, stride):
    """
    计算卷积层输出特征图的尺寸
    
    参数:
        input_channels: 输入通道数
        input_height: 输入高度
        input_width: 输入宽度
        num_kernels: 卷积核数量
        kernel_channels: 每个卷积核的通道数
        kernel_height: 卷积核高度
        kernel_width: 卷积核宽度
        padding: 填充大小
        stride: 步幅大小
    
    返回:
        output_channels: 输出通道数
        output_height: 输出高度
        output_width: 输出宽度
    """
    output_channels = num_kernels
    output_height = (input_height + 2 * padding - kernel_height) // stride + 1
    output_width = (input_width + 2 * padding - kernel_width) // stride + 1
    
    return output_channels, output_height, output_width


def calculate_multiplications_per_pixel(input_channels, kernel_height, kernel_width):
    """
    计算单个输出像素所需的点乘（乘法）操作次数
    
    参数:
        input_channels: 输入通道数
        kernel_height: 卷积核高度
        kernel_width: 卷积核宽度
    
    返回:
        num_multiplications: 乘法操作次数
    """
    return input_channels * kernel_height * kernel_width


# 题目参数
input_channels = 3
input_height = 32
input_width = 32

num_kernels = 16
kernel_channels = 3  # 必须等于输入通道数
kernel_height = 5
kernel_width = 5

padding = 2
stride = 2

# 计算输出特征图尺寸
output_channels, output_height, output_width = calculate_convolution_output(
    input_channels, input_height, input_width,
    num_kernels, kernel_channels, kernel_height, kernel_width,
    padding, stride
)

# 计算单个输出像素的乘法次数
multiplications_per_pixel = calculate_multiplications_per_pixel(
    input_channels, kernel_height, kernel_width
)

# 打印结果
print("=== 卷积计算结果 ===")
print(f"输入图像尺寸: {input_channels} × {input_height} × {input_width}")
print(f"卷积核配置: {num_kernels} 个, 每个大小 {kernel_channels} × {kernel_height} × {kernel_width}")
print(f"填充 (Padding): {padding}")
print(f"步幅 (Stride): {stride}")
print()
print("1. 输出特征图尺寸: {} × {} × {}".format(output_channels, output_height, output_width))
print("2. 单个输出像素的点乘次数: {}".format(multiplications_per_pixel))
print()
print("=" * 40)
print("计算原理说明")
print("=" * 40)
print("输出高度/宽度的计算公式:")
print("  output_height = floor((input_height + 2*padding - kernel_height) / stride) + 1")
print()
print("代入数值:")
print("  output_height = floor((32 + 2*2 - 5) / 2) + 1 = floor(31/2) + 1 = 15 + 1 = 16")
print()
print("点乘次数计算:")
print("  每个通道需要 5 × 5 = 25 次乘法")
print("  3 个通道总共需要 3 × 5 × 5 = 75 次乘法")


=== 卷积计算结果 ===
输入图像尺寸: 3 × 32 × 32
卷积核配置: 16 个, 每个大小 3 × 5 × 5
填充 (Padding): 2
步幅 (Stride): 2

1. 输出特征图尺寸: 16 × 16 × 16
2. 单个输出像素的点乘次数: 75

计算原理说明
输出高度/宽度的计算公式:
  output_height = floor((input_height + 2*padding - kernel_height) / stride) + 1

代入数值:
  output_height = floor((32 + 2*2 - 5) / 2) + 1 = floor(31/2) + 1 = 15 + 1 = 16

点乘次数计算:
  每个通道需要 5 × 5 = 25 次乘法
  3 个通道总共需要 3 × 5 × 5 = 75 次乘法


In [2]:
import numpy as np

def max_pool2d(input_data, kernel_size=2, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播函数
    
    参数:
        input_data: 输入数据，形状为 (channels, height, width)
        kernel_size: 池化核大小，整数或元组 (kh, kw)
        stride: 步幅，整数或元组 (sh, sw)
        padding: 填充大小，整数或元组 (ph, pw)
    
    返回:
        output: 输出数据，形状为 (channels, output_height, output_width)
    """
    # 处理参数
    if isinstance(kernel_size, int):
        kernel_height = kernel_width = kernel_size
    else:
        kernel_height, kernel_width = kernel_size
    
    if isinstance(stride, int):
        stride_height = stride_width = stride
    else:
        stride_height, stride_width = stride
    
    if isinstance(padding, int):
        padding_height = padding_width = padding
    else:
        padding_height, padding_width = padding
    
    # 获取输入形状
    channels, input_height, input_width = input_data.shape
    
    # 计算输出尺寸
    output_height = (input_height + 2 * padding_height - kernel_height) // stride_height + 1
    output_width = (input_width + 2 * padding_width - kernel_width) // stride_width + 1
    
    # 初始化输出
    output = np.zeros((channels, output_height, output_width))
    
    # 对每个通道进行池化
    for c in range(channels):
        # 对输出的每个位置进行计算
        for oh in range(output_height):
            for ow in range(output_width):
                # 计算感受野在输入上的位置
                ih_start = oh * stride_height - padding_height
                ih_end = ih_start + kernel_height
                iw_start = ow * stride_width - padding_width
                iw_end = iw_start + kernel_width
                
                # 获取感受野区域（处理边界情况）
                region = []
                for ih in range(max(0, ih_start), min(input_height, ih_end)):
                    for iw in range(max(0, iw_start), min(input_width, iw_end)):
                        region.append(input_data[c, ih, iw])
                
                # 如果区域为空（padding导致），取0
                if len(region) == 0:
                    output[c, oh, ow] = 0
                else:
                    # 取最大值
                    output[c, oh, ow] = np.max(region)
    
    return output


def test_max_pool2d():
    """
    测试最大池化函数
    """
    print("\n" + "=" * 40)
    print("2.2 编程题：最大池化实现测试")
    print("=" * 40)
    
    # 创建测试输入 (2通道, 4x4)
    np.random.seed(42)
    test_input = np.random.randint(0, 10, size=(2, 4, 4))
    print("测试输入 (2通道, 4x4):")
    for c in range(test_input.shape[0]):
        print(f"通道 {c}:")
        print(test_input[c])
        print()
    
    # 测试1: 默认参数 (kernel=2, stride=1, padding=0)
    print("测试1: kernel_size=2, stride=1, padding=0")
    output1 = max_pool2d(test_input, kernel_size=2, stride=1, padding=0)
    print(f"输出形状: {output1.shape}")
    for c in range(output1.shape[0]):
        print(f"通道 {c}:")
        print(output1[c])
        print()
    
    # 测试2: kernel=2, stride=2, padding=0
    print("测试2: kernel_size=2, stride=2, padding=0")
    output2 = max_pool2d(test_input, kernel_size=2, stride=2, padding=0)
    print(f"输出形状: {output2.shape}")
    for c in range(output2.shape[0]):
        print(f"通道 {c}:")
        print(output2[c])
        print()
    
    # 测试3: kernel=3, stride=1, padding=1
    print("测试3: kernel_size=3, stride=1, padding=1")
    output3 = max_pool2d(test_input, kernel_size=3, stride=1, padding=1)
    print(f"输出形状: {output3.shape}")
    for c in range(output3.shape[0]):
        print(f"通道 {c}:")
        print(output3[c])
        print()


# 运行测试
test_max_pool2d()


2.2 编程题：最大池化实现测试
测试输入 (2通道, 4x4):
通道 0:
[[6 3 7 4]
 [6 9 2 6]
 [7 4 3 7]
 [7 2 5 4]]

通道 1:
[[1 7 5 1]
 [4 0 9 5]
 [8 0 9 2]
 [6 3 8 2]]

测试1: kernel_size=2, stride=1, padding=0
输出形状: (2, 3, 3)
通道 0:
[[9. 9. 7.]
 [9. 9. 7.]
 [7. 5. 7.]]

通道 1:
[[7. 9. 9.]
 [8. 9. 9.]
 [8. 9. 9.]]

测试2: kernel_size=2, stride=2, padding=0
输出形状: (2, 2, 2)
通道 0:
[[9. 7.]
 [7. 7.]]

通道 1:
[[7. 9.]
 [8. 9.]]

测试3: kernel_size=3, stride=1, padding=1
输出形状: (2, 4, 4)
通道 0:
[[9. 9. 9. 7.]
 [9. 9. 9. 7.]
 [9. 9. 9. 7.]
 [7. 7. 7. 7.]]

通道 1:
[[7. 9. 9. 9.]
 [8. 9. 9. 9.]
 [8. 9. 9. 9.]
 [8. 9. 9. 9.]]



In [3]:
def calculate_params_5x5(input_channels, output_channels):
    """
    计算一个 5x5 卷积层（不带偏置）的参数量
    
    参数:
        input_channels: 输入通道数
        output_channels: 输出通道数
    
    返回:
        params: 参数量
    """
    # 每个卷积核有 5x5xinput_channels 个参数
    # 共有 output_channels 个卷积核
    return output_channels * 5 * 5 * input_channels


def calculate_params_two_3x3(input_channels, hidden_channels, output_channels):
    """
    计算两个串联的 3x3 卷积层（不带偏置）的总参数量
    
    参数:
        input_channels: 输入通道数
        hidden_channels: 中间通道数（第一层输出通道数）
        output_channels: 输出通道数（第二层输出通道数）
    
    返回:
        total_params: 总参数量
    """
    # 第一个 3x3 卷积层
    layer1_params = hidden_channels * 3 * 3 * input_channels
    # 第二个 3x3 卷积层
    layer2_params = output_channels * 3 * 3 * hidden_channels
    return layer1_params + layer2_params


def test_vgg_params():
    """
    测试 VGG 卷积核参数量计算
    """
    print("\n" + "=" * 40)
    print("3.1 理论计算题：VGG 卷积核参数量比较")
    print("=" * 40)
    
    # 题目条件：输入和输出通道数均为 C
    C = 64  # 假设 C = 64（VGG 常用通道数）
    
    print(f"假设通道数 C = {C}")
    print()
    
    # 1. 计算一个 5x5 卷积层的参数量
    params_5x5 = calculate_params_5x5(C, C)
    print(f"1. 一个 5×5 卷积层的参数量: {params_5x5:,}")
    
    # 2. 计算两个串联的 3x3 卷积层的总参数量（两层通道数都为 C）
    params_two_3x3 = calculate_params_two_3x3(C, C, C)
    print(f"2. 两个串联的 3×3 卷积层的总参数量: {params_two_3x3:,}")
    
    # 计算参数量减少比例
    reduction_ratio = (params_5x5 - params_two_3x3) / params_5x5 * 100
    print()
    print(f"参数量减少比例: {reduction_ratio:.2f}%")
    print()
    
    # 计算感受野
    print("感受野分析:")
    print(f"  一个 5×5 卷积的感受野: 5×5")
    print(f"  两个串联 3×3 卷积的感受野: 5×5 (与单个 5×5 相同)")
    print()
    print("结论: 使用两个 3×3 卷积可以在保持相同感受野的情况下，显著减少参数量")


# 运行测试
test_vgg_params()


3.1 理论计算题：VGG 卷积核参数量比较
假设通道数 C = 64

1. 一个 5×5 卷积层的参数量: 102,400
2. 两个串联的 3×3 卷积层的总参数量: 73,728

参数量减少比例: 28.00%

感受野分析:
  一个 5×5 卷积的感受野: 5×5
  两个串联 3×3 卷积的感受野: 5×5 (与单个 5×5 相同)

结论: 使用两个 3×3 卷积可以在保持相同感受野的情况下，显著减少参数量


In [4]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义一个标准的 NiN 块（NiN Block）
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 卷积核大小
        stride: 步幅
        padding: 填充
    
    返回:
        nn.Sequential: NiN 块
    """
    return nn.Sequential(
        # 第一个卷积层（普通卷积）
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        
        # 第一个 1x1 卷积层
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        
        # 第二个 1x1 卷积层
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )


def test_nin_block():
    """
    测试 NiN 块
    """
    print("\n" + "=" * 40)
    print("3.2 编程题：NiN 块实现测试")
    print("=" * 40)
    
    # 创建一个 NiN 块实例
    # 输入通道数 3，输出通道数 96，卷积核大小 11，步幅 4，填充 0（AlexNet 风格）
    nin = nin_block(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
    
    print("NiN 块结构:")
    print(nin)
    print()
    
    # 创建测试输入 (batch_size=1, channels=3, height=224, width=224)
    test_input = torch.randn(1, 3, 224, 224)
    print(f"测试输入形状: {test_input.shape}")
    
    # 前向传播
    output = nin(test_input)
    print(f"测试输出形状: {output.shape}")
    print()
    
    # 计算参数量
    total_params = sum(p.numel() for p in nin.parameters())
    print(f"NiN 块参数量: {total_params:,}")
    
    # 与传统全连接层对比
    # 如果用全连接层将 54x54x96 的特征图展平
    fc_params = (54 * 54 * 96) * 96  # 假设输出也是 96 维
    print(f"等效全连接层参数量: {fc_params:,}")
    print(f"参数量减少比例: {(1 - total_params / fc_params) * 100:.2f}%")


# 运行测试
test_nin_block()


3.2 编程题：NiN 块实现测试
NiN 块结构:
Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

测试输入形状: torch.Size([1, 3, 224, 224])
测试输出形状: torch.Size([1, 96, 54, 54])

NiN 块参数量: 53,568
等效全连接层参数量: 26,873,856
参数量减少比例: 99.80%


In [5]:
def batch_normalization(x, gamma, beta, epsilon=0):
    """
    计算 Batch Normalization 的前向传播
    
    参数:
        x: 输入数据列表
        gamma: 缩放参数
        beta: 平移参数
        epsilon: 常数，用于数值稳定性
    
    返回:
        y: 归一化后的输出列表
    """
    # 计算均值
    mu = sum(x) / len(x)
    
    # 计算方差
    variance = sum((xi - mu) ** 2 for xi in x) / len(x)
    
    # 计算标准差
    std = (variance + epsilon) ** 0.5
    
    # 归一化
    x_hat = [(xi - mu) / std for xi in x]
    
    # 缩放和平移
    y = [gamma * xi + beta for xi in x_hat]
    
    return y, mu, variance, x_hat


def test_batch_norm():
    """
    测试 Batch Normalization 计算
    """
    print("\n" + "=" * 40)
    print("4.1 理论计算题：Batch Normalization 计算")
    print("=" * 40)
    
    # 题目参数
    x = [2, 4, 6, 8]  # 4个样本的输出值
    gamma = 2  # 缩放参数
    beta = 1   # 平移参数
    epsilon = 0  # 常数
    
    print(f"输入值: {x}")
    print(f"缩放参数 γ = {gamma}")
    print(f"平移参数 β = {beta}")
    print(f"常数 ε = {epsilon}")
    print()
    
    # 计算 Batch Normalization
    y, mu, variance, x_hat = batch_normalization(x, gamma, beta, epsilon)
    
    # 打印中间结果
    print("计算过程:")
    print(f"1. 均值 μ = (2 + 4 + 6 + 8) / 4 = {mu}")
    print(f"2. 方差 σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4 = {variance}")
    print(f"3. 标准差 σ = √{variance} = {(variance ** 0.5):.2f}")
    print()
    
    print("4. 归一化值 x̂:")
    for i, (xi, xi_hat) in enumerate(zip(x, x_hat), 1):
        print(f"   x{i}_hat = ({xi} - {mu}) / {(variance ** 0.5):.2f} = {xi_hat:.2f}")
    print()
    
    print("5. 缩放和平移 y = γ * x̂ + β:")
    for i, (xi_hat, yi) in enumerate(zip(x_hat, y), 1):
        print(f"   y{i} = {gamma} * {xi_hat:.2f} + {beta} = {yi:.2f}")
    print()
    
    # 打印最终结果
    print("最终输出:")
    print(f"y1 = {y[0]:.2f}, y2 = {y[1]:.2f}, y3 = {y[2]:.2f}, y4 = {y[3]:.2f}")


# 运行测试
test_batch_norm()


4.1 理论计算题：Batch Normalization 计算
输入值: [2, 4, 6, 8]
缩放参数 γ = 2
平移参数 β = 1
常数 ε = 0

计算过程:
1. 均值 μ = (2 + 4 + 6 + 8) / 4 = 5.0
2. 方差 σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4 = 5.0
3. 标准差 σ = √5.0 = 2.24

4. 归一化值 x̂:
   x1_hat = (2 - 5.0) / 2.24 = -1.34
   x2_hat = (4 - 5.0) / 2.24 = -0.45
   x3_hat = (6 - 5.0) / 2.24 = 0.45
   x4_hat = (8 - 5.0) / 2.24 = 1.34

5. 缩放和平移 y = γ * x̂ + β:
   y1 = 2 * -1.34 + 1 = -1.68
   y2 = 2 * -0.45 + 1 = 0.11
   y3 = 2 * 0.45 + 1 = 1.89
   y4 = 2 * 1.34 + 1 = 3.68

最终输出:
y1 = -1.68, y2 = 0.11, y3 = 1.89, y4 = 3.68


In [7]:
class Residual(nn.Module):
    """
    自定义残差块类
    
    参数:
        input_channels: 输入通道数
        output_channels: 输出通道数
        use_1x1conv: 是否使用 1x1 卷积来调整输入形状
        stride: 步幅（默认 1）
    """
    def __init__(self, input_channels, output_channels, use_1x1conv=False, stride=1):
        super().__init__()
        
        # 第一个 3x3 卷积层
        self.conv1 = nn.Conv2d(input_channels, output_channels, 
                               kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(output_channels)
        
        # 第二个 3x3 卷积层
        self.conv2 = nn.Conv2d(output_channels, output_channels, 
                               kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(output_channels)
        
        # 如果需要调整输入通道数和形状，使用 1x1 卷积
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, output_channels, 
                                   kernel_size=1, stride=stride)
        
        # ReLU 激活函数
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入张量
        
        返回:
            残差块输出
        """
        # 主路径：conv1 -> bn1 -> relu -> conv2 -> bn2
        Y = self.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        
        # 短路连接路径
        if self.conv3 is not None:
            X = self.conv3(X)
        
        # 残差连接：f(x) + x
        Y += X
        
        # 最后的 ReLU 激活
        return self.relu(Y)


def test_residual_block():
    """
    测试残差块
    """
    print("\n" + "=" * 40)
    print("4.2 编程题：ResNet 残差块实现测试")
    print("=" * 40)
    
    # 测试1: 不使用 1x1 卷积（输入输出通道相同）
    print("测试1: 不使用 1x1 卷积 (input_channels=64, output_channels=64)")
    res1 = Residual(64, 64)
    print(res1)
    test_input1 = torch.randn(1, 64, 32, 32)
    output1 = res1(test_input1)
    print(f"输入形状: {test_input1.shape}")
    print(f"输出形状: {output1.shape}")
    print()
    
    # 测试2: 使用 1x1 卷积（输入输出通道不同）
    print("测试2: 使用 1x1 卷积 (input_channels=64, output_channels=128, stride=2)")
    res2 = Residual(64, 128, use_1x1conv=True, stride=2)
    print(res2)
    test_input2 = torch.randn(1, 64, 32, 32)
    output2 = res2(test_input2)
    print(f"输入形状: {test_input2.shape}")
    print(f"输出形状: {output2.shape}")
    print()
    
    # 测试3: 使用 1x1 卷积且改变步幅（输入输出通道相同但空间尺寸变化）
    print("测试3: 使用 1x1 卷积且改变步幅 (input_channels=64, output_channels=64, stride=2)")
    res3 = Residual(64, 64, use_1x1conv=True, stride=2)
    print(res3)
    test_input3 = torch.randn(1, 64, 32, 32)
    output3 = res3(test_input3)
    print(f"输入形状: {test_input3.shape}")
    print(f"输出形状: {output3.shape}")


# 运行测试
test_residual_block()


4.2 编程题：ResNet 残差块实现测试
测试1: 不使用 1x1 卷积 (input_channels=64, output_channels=64)
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)
输入形状: torch.Size([1, 64, 32, 32])
输出形状: torch.Size([1, 64, 32, 32])

测试2: 使用 1x1 卷积 (input_channels=64, output_channels=128, stride=2)
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(64,

In [ ]:
#1. 为什么对底层特征提取层设置较小学习率，对顶层输出层设置较大学习率？
原因如下：

底层特征的通用性： 预训练模型的底层卷积层学习到的是通用特征（如边缘、纹理、简单形状等），这些特征在不同任务中都很有用，已经是高质量的特征表示，不需要大幅调整。
顶层特征的任务特异性： 模型的顶层（尤其是输出层）学习到的是与源数据集高度相关的特定特征。在微调时，我们需要将这些特征适应到新的目标任务，因此需要更大的学习率来快速更新。
参数初始化差异： 新初始化的顶层输出层参数是随机的，需要快速学习目标任务的分类边界；而底层参数已经在大规模数据上训练过，包含有价值的先验知识，应该保留。
防止灾难性遗忘： 如果对底层使用较大学习率，可能会破坏已学习的通用特征，导致模型性能下降。
## 2. 目标数据集很小且与源数据集相似时的微调策略
当目标数据集很小且与源数据集相似时，主要风险是过拟合，推荐策略：

冻结策略：

完全冻结底层： 冻结除最后几层外的所有预训练参数，只训练新添加的输出层。
分层微调： 对不同层设置不同学习率，底层学习率很小或为0，越靠近输出层学习率越大。
数据增强：

对目标数据集进行强烈的数据增强（随机裁剪、翻转、颜色扰动等），增加数据多样性。
正则化：

使用较大的权重衰减（Weight Decay）。
添加 Dropout 层。
使用早停（Early Stopping）策略。
模型选择：

选择较小的模型或减少训练轮数。
使用预训练模型的中间层特征进行特征提取，而不是端到端训练。
迁移学习方式：

如果任务非常相似，可以只替换输出层，完全不微调底层。
使用特征提取（Feature Extraction）模式，而不是完整微调。
核心原则： 目标数据集越小、与源数据集越相似，就应该冻结越多的底层参数，避免过度拟合到有限的目标数据上。

In [15]:
# ================================================
# 5.2 编程题：图像增广管道实现
# ================================================

from torchvision import transforms
from PIL import Image
import numpy as np

def create_augmentation_pipeline():
    """
    创建组合图像增广管道
    
    包含以下操作：
    1. 随机裁剪，面积比例在 0.08 到 1.0 之间，缩放到 224×224
    2. 50% 的概率水平翻转
    3. 随机改变亮度、对比度和饱和度，变化范围为 0.5
    4. 转换为 PyTorch 张量
    """
    transform = transforms.Compose([
        # 1. 随机裁剪，面积比例在 0.08 到 1.0 之间，缩放到 224x224
        transforms.RandomResizedCrop(
            size=(224, 224),
            scale=(0.08, 1.0)
        ),
        
        # 2. 50% 的概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        
        # 3. 随机改变亮度、对比度和饱和度，变化范围为 0.5
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5
        ),
        
        # 4. 转换为 PyTorch 张量
        transforms.ToTensor()
    ])
    
    return transform

# 测试代码
if __name__ == '__main__':
    # 创建图像增广管道
    transform = create_augmentation_pipeline()
    
    # 创建测试图像
    np.random.seed(42)
    test_image = Image.fromarray(np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8))
    
    # 应用增广管道
    augmented_tensor = transform(test_image)
    print(f'增广后形状: {augmented_tensor.shape}')

增广后形状: torch.Size([3, 224, 224])


In [16]:
# 验证 IoU 计算
def calculate_iou(box_a, box_b):
    # 提取坐标
    x1_a, y1_a, x2_a, y2_a = box_a
    x1_b, y1_b, x2_b, y2_b = box_b
    
    # 计算交集区域
    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)
    
    # 计算交集面积
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    
    # 计算各自面积
    area_a = (x2_a - x1_a) * (y2_a - y1_a)
    area_b = (x2_b - x1_b) * (y2_b - y1_b)
    
    # 计算并集面积
    union_area = area_a + area_b - inter_area
    
    # 计算 IoU
    iou = inter_area / union_area
    return iou

# 测试
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
iou = calculate_iou(A, B)
print(f"IoU = {iou:.4f} ({iou})")

IoU = 0.1429 (0.14285714285714285)


In [18]:
import torch
import torch.nn.functional as F

# ================================================
# 1. 先定义标签平滑交叉熵损失函数
# ================================================
def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失
    
    参数：
        logits: 模型输出的 logits，形状为 [batch_size, num_classes]
        targets: 真实标签，形状为 [batch_size]，值为类别索引
        epsilon: 平滑因子，默认 0.1
    
    返回：
        平均交叉熵损失
    """
    num_classes = logits.size(-1)
    smooth_targets = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)
    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_targets * log_probs).sum(dim=1).mean()
    return loss

# ================================================
# 2. 再调用函数进行测试
# ================================================
if __name__ == '__main__':
    # 输入数据
    logits = torch.tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229],
                           [-0.1863,  2.2082, -0.6380,  0.4617,  0.2674]])
    targets = torch.tensor([2, 0])
    
    # 计算损失
    loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)
    standard_loss = F.cross_entropy(logits, targets)
    
    # 输出结果
    print(f'标签平滑交叉熵损失: {loss.item():.6f}')
    print(f'标准交叉熵损失: {standard_loss.item():.6f}')

标签平滑交叉熵损失: 2.092640
标准交叉熵损失: 2.113633
